### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# ! vectorstores
from langchain_community.vectorstores import FAISS, Chroma, Pinecone, Weaviate, Milvus, Qdrant


## utility imports
import numpy as np
from typing import List

In [2]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [3]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    eff

In [4]:
import tempfile
temp_dir = tempfile.mkdtemp()
print(f"Temporary directory created at: {temp_dir}")
for i, doc in enumerate(sample_docs):
    file_path = f"docs/doc_{i+1}.txt"
    with open(file_path, 'w') as f:
        f.write(doc)
    print(f"Document {i+1} saved at: {file_path}")
    

Temporary directory created at: C:\Users\ARNAVB~1\AppData\Local\Temp\tmp8fmud31o
Document 1 saved at: docs/doc_1.txt
Document 2 saved at: docs/doc_2.txt
Document 3 saved at: docs/doc_3.txt


In [5]:
from langchain_community.document_loaders import TextLoader,DirectoryLoader

loader=DirectoryLoader("docs", glob="*.txt", )
documents=loader.load()
print(f"Number of documents loaded: {len(documents)}")
print("Sample document content:")
print(documents[0].page_content[:500])  # Print the first 500 characters of the first document


Number of documents loaded: 3
Sample document content:
Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewar


## Document Splitting

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50,length_function=len)
# text_splitter.split_documents(documents)
chunks=text_splitter.split_documents(documents)
print(f"Number of chunks created: {len(chunks)}")
print("Sample chunk content:")
print(chunks[0].page_content[:500])  # Print the first 500 characters of the first chunk

Number of chunks created: 5
Sample chunk content:
Machine Learning Fundamentals


In [7]:
chunks

[Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'docs\\doc_2.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'docs\\doc_2.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep le

## Embedding MOdels

In [8]:
sample_text="Machine Learning is facinating"
embeddings=OllamaEmbeddings(model="llama3.2:latest")
embeddings


OllamaEmbeddings(model='llama3.2:latest', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [9]:
embeddings.embed_query(sample_text)

[-0.012043369,
 0.016861685,
 0.04180385,
 -0.014466409,
 -0.020095624,
 -0.04560012,
 0.018800316,
 0.0004905143,
 -0.013566924,
 -0.015364271,
 -0.008641946,
 -0.020964768,
 0.0074825077,
 0.03223642,
 0.0058638556,
 -0.011616594,
 0.008150148,
 0.0023836666,
 0.00086505245,
 0.0061067375,
 0.0021974372,
 -0.021860462,
 0.029815856,
 -0.017236723,
 0.0076064547,
 -0.0153949605,
 0.023596995,
 -0.040058948,
 0.017899202,
 0.004671267,
 -0.009727652,
 -0.006471602,
 0.009034995,
 -0.0053049237,
 0.03227416,
 0.00048502005,
 -0.015607403,
 0.028760463,
 -0.0101156235,
 -0.023348317,
 -0.018770183,
 -0.01744114,
 0.011482883,
 0.0046852655,
 -0.018003376,
 0.0052916748,
 0.0066463216,
 0.014486938,
 0.011929245,
 -0.02734766,
 0.0077267885,
 0.017156003,
 0.020057231,
 0.010966691,
 0.0014332853,
 0.006344407,
 0.015087748,
 -0.028868567,
 0.007870271,
 0.019556755,
 0.0065760785,
 0.0037592635,
 0.026607724,
 -0.0064209765,
 0.0298617,
 -0.0754462,
 -0.017583137,
 -0.0062064254,
 -0.004

## Initialize the Chroma DB

In [10]:
persistance_directory="./chroma_db"
chroma_db=Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory=persistance_directory,collection_name="rag_collection")

print("Persisting ChromaDB to disk...")
chroma_db.persist()
print(f"ChromaDB persisted successfully at: {persistance_directory}")
print(chroma_db._collection.count())


Persisting ChromaDB to disk...
ChromaDB persisted successfully at: ./chroma_db
13


C:\Users\ArnavBhatia\AppData\Local\Temp\ipykernel_2360\4266146322.py:5: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_db.persist()


Similarity Search

In [11]:
query="What is machine learning?"
similar_docs=chroma_db.similarity_search(query,k=2)
print(f"Number of similar documents retrieved: {len(similar_docs)}")
similar_docs


Number of similar documents retrieved: 2


[Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled 

In [12]:
query="What is NLP?"
similar_docs=chroma_db.similarity_search(query,k=2)
print(f"Number of similar documents retrieved: {len(similar_docs)}")


Number of similar documents retrieved: 2


In [13]:
query="What is Deep Learining?"
similar_docs=chroma_db.similarity_search(query,k=2)
print(f"Number of similar documents retrieved: {len(similar_docs)}")


Number of similar documents retrieved: 2


In [14]:
for i ,doc in enumerate(similar_docs):
    print(f"Similar Document {i+1} Content:")
    print(doc.page_content)
    print("-" * 50)
    print(f"Metadata: {doc.metadata}")

Similar Document 1 Content:
Reinforcement Learning in Detail
--------------------------------------------------
Metadata: {'source': 'new_doc.txt'}
Similar Document 2 Content:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.
--------------------------------------------------
Metadata: {'source': 'docs\\doc_1.txt'}


### Advance Similarity Search

In [15]:
results_scores=chroma_db.similarity_search_with_score(query,k=2)
results_scores

[(Document(metadata={'source': 'new_doc.txt'}, page_content='Reinforcement Learning in Detail'),
  0.7125671276237034),
 (Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'),
  0.7129333372988603)]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

### Initialize LLM,RAG,Prompt Technique, Query the RAG System

In [16]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:latest")
llm.invoke("Hello, how are you?")


AIMessage(content="I'm just a language model, so I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have! How can I help you today?", additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-24T17:55:13.1361523Z', 'done': True, 'done_reason': 'stop', 'total_duration': 750996000, 'load_duration': 137021600, 'prompt_eval_count': 31, 'prompt_eval_duration': 42135600, 'eval_count': 47, 'eval_duration': 516412800, 'model_name': 'llama3.2:latest'}, id='lc_run--019dc0a1-a579-7bd2-8ec8-bae47633316b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 47, 'total_tokens': 78})

In [17]:
test_response=llm.invoke("What is LLM?")
test_response

AIMessage(content="LLM stands for Large Language Model. It's a type of artificial intelligence (AI) model that's specifically designed to process and understand human language. Large Language Models are trained on vast amounts of text data, which enables them to learn patterns, relationships, and context in language.\n\nThe primary goal of LLMs is to generate human-like text based on the input they receive. They can be used for a variety of tasks, such as:\n\n1. Text generation: LLMs can create new text that's coherent and readable.\n2. Language translation: LLMs can translate text from one language to another.\n3. Question answering: LLMs can answer questions based on the context provided.\n4. Summarization: LLMs can summarize long pieces of text into shorter, more digestible versions.\n\nLLMs are particularly useful in applications such as:\n\n1. Virtual assistants: LLMs power virtual assistants like Siri, Alexa, and Google Assistant.\n2. Chatbots: LLMs enable chatbots to engage with

In [18]:
from langchain.chat_models.base import init_chat_model
llm = init_chat_model("ollama:llama3.2:latest")
llm.invoke("What is RAG?")

AIMessage(content="RAG can have different meanings depending on the context. Here are a few possible interpretations:\n\n1. Resource Allocation Group: In this context, RAG refers to a team or group responsible for managing and allocating resources within an organization.\n2. Ready-to-Use Gas (RAG) mixture: In the oil and gas industry, RAG is a type of mixture used as a substitute for natural gas when it's not available or too expensive to use.\n3. Red Adenocarcinoma Gene (RAG): In genetics, RAG refers to a gene that plays a crucial role in the development and function of the immune system.\n4. Real-time Graphics: In computing and graphics, RAG can refer to real-time graphics rendering engines or libraries used for creating immersive gaming experiences.\n\nIf you could provide more context or information about where you encountered this term, I'd be happy to give you a more specific answer.", additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-24T

##  Modern Rag Chain 

In [19]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

retriever=chroma_db.as_retriever(search_kwargs={"k":3})
system_prompt="You are a helpful assistant that provides concise answers based on retrieved documents. {context}"
prompt=ChatPromptTemplate.from_messages([("system",system_prompt),("human","{input}")])

combine_docs_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)
print(combine_docs_chain)



bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a helpful assistant that provides concise answers based on retrieved documents. {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(model='llama3.2:latest')
| StrOutputParser() kwargs={} config={'run_name': 'stuff_documents_chain'} config_factories=[]


This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

retrieval

In [20]:
retrieval_chain=create_retrieval_chain(retriever,combine_docs_chain)
query="What is NLP?"
response=retrieval_chain.invoke({"input": query})

In [21]:
print("Final Response:")
for i,j in response.items():
    print(f"{i}: {j}")


Final Response:
input: What is NLP?
context: [Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'), Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupe

In [22]:
import langchain
langchain.__version__

'1.2.12'

### Rag Alternative LCEL

In [23]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [24]:
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [25]:
def format_docs(docs: List[Document]):
    return "\n\n".join([f"Document {i+1}:\n{doc.page_content}" for i, doc in enumerate(docs)])

In [26]:
rag_chain_lcel=(
    {"context": retriever|format_docs,"question": RunnablePassthrough()}
      | custom_prompt | llm | StrOutputParser()
)

In [27]:
answer=rag_chain_lcel.invoke("what is NLP?")

In [28]:
print("Final Response:")
print(answer)

Final Response:
NLP stands for Natural Language Processing. 

Specific details from the context that support this answer include:

* Document 3 specifically mentions NLP as a field of AI that focuses on the interaction between computers and human language.
* It lists key tasks in NLP, including text classification, named entity recognition, sentiment analysis, machine translation, and question answering, which further supports its definition.


In [29]:
def query_lcel_chain(question):
    print(f"Query: {question}")
    answer=rag_chain_lcel.invoke(question)
    print("Final Response:")
    print(answer)

    # docs=retriever.get_relevant_documents(question)
    docs=retriever.invoke(question)
    print(f"Retrieved {len(docs)} relevant documents:")
    for i, doc in enumerate(docs):
        print(f"Document {i+1} Content:")
        print(doc.page_content)
        print("-" * 50)
        print(f"Metadata: {doc.metadata}")
        

In [30]:
query_lcel_chain("What is Deep Learning?")

Query: What is Deep Learning?
Final Response:
I can provide an answer based on Document 2 and Document 3.

According to the context, particularly Document 2 and Document 3 which both mention "Deep Learning and Neural Networks", I can infer that:

Deep Learning refers to a subset of Machine Learning approaches that rely on multiple layers (or "deep" networks) of artificial neural networks. These networks are designed to learn complex patterns in data by automatically adjusting the connections between them.

The specific details from Document 2 and Document 3 support this answer, as they both discuss Deep Learning and Neural Networks in the same context, implying a connection between the two concepts.
Retrieved 3 relevant documents:
Document 1 Content:
Reinforcement Learning in Detail
--------------------------------------------------
Metadata: {'source': 'new_doc.txt'}
Document 2 Content:
Deep Learning and Neural Networks
--------------------------------------------------
Metadata: {'so

In [31]:
query_lcel_chain("What are key concepts in Reinforcement Learning?")

Query: What are key concepts in Reinforcement Learning?
Final Response:
According to Document 3, the key concepts in Reinforcement Learning are:

1. Using interaction with an environment
2. Rewards and penalties

Document 3 mentions that reinforcement learning learns through these interactions, which is further supported by the fact that RL has been successfully applied to game playing (like AlphaGo), robotics, and autonomous systems, as mentioned in Document 1.

It does not specify additional key concepts beyond rewards and penalties, but it does acknowledge its application to real-world scenarios.
Retrieved 3 relevant documents:
Document 1 Content:
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
--------------------------------------------------
Metadata: {'source': 'new_doc.txt'}
Document 2 Content:
Reinforcement Learning in Detail
--------------------------------------------------
Metadata: {'source': 'new_do

## Add new documents to an Existing vectorstore

In [32]:
chroma_db

In [33]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [34]:
new_doc = Document(page_content=new_document, metadata={"source": "new_doc.txt"})
new_doc

Document(metadata={'source': 'new_doc.txt'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [35]:
new_chunks=text_splitter.split_documents([new_doc])
print(f"Number of new chunks created: {len(new_chunks)}")


Number of new chunks created: 3


In [36]:
chroma_db.add_documents(new_chunks)
chroma_db.persist()
print("New chunks added to ChromaDB.")

New chunks added to ChromaDB.


querry with updated vectorstore

In [37]:
query_lcel_chain("What are the key concepts in Reinforcement Learning?")

Query: What are the key concepts in Reinforcement Learning?
Final Response:
The key concepts in Reinforcement Learning mentioned in Document 3 are:

1. Rewards
2. Penalties

These two concepts are used in conjunction with each other to learn through interaction with an environment. According to Document 3, "Reinforcement learning learns through interaction with an environment using rewards and penalties."

Additionally, the context mentions that Reinforcement Learning is a subset of Machine Learning, which further supports this answer.

It's worth noting that supervised learning and unsupervised learning are also mentioned in Document 3 as other types of machine learning, but they are not directly related to reinforcement learning.
Retrieved 3 relevant documents:
Document 1 Content:
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
--------------------------------------------------
Metadata: {'source': 'new_doc.txt

### Advance Rag Techniques - Conversational Memory

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents
- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage



In [42]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [43]:
history_aware_retriever = create_history_aware_retriever(
    llm,retriever,contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001431AE40E10>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(ta

In [44]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [45]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")


Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed, allowing them to make predictions or decisions based on data patterns. It has three main types: supervised learning, unsupervised learning, and reinforcement learning. Machine learning helps systems adapt to new information and improve their performance over time.


In [46]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [47]:
chat_history

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed, allowing them to make predictions or decisions based on data patterns. It has three main types: supervised learning, unsupervised learning, and reinforcement learning. Machine learning helps systems adapt to new information and improve their performance over time.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [48]:
# ! follow up question

result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"})
print(f"Q: What are its main types?")
print(f"A: {result2['answer']}")


Q: What are its main types?
A: The main types of machine learning are Supervised Learning, Unsupervised Learning, and Reinforcement Learning. 

1. Supervised Learning: involves training on labeled data to make predictions.
2. Unsupervised Learning: discovers patterns in unlabeled data without prior knowledge of the target variable.
3. Reinforcement Learning: learns through trial and error by interacting with an environment and receiving rewards or penalties for actions taken.


## Groq Models

In [50]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
groq_llm=ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api_key)

groq_response=groq_llm.invoke("What is RAG?")
print("Groq LLM Response:")

Groq LLM Response:


In [51]:
groq_response

AIMessage(content="RAG can refer to several things depending on the context, but here are a few possible meanings:\n\n1. **RAG (Roller derby)**: In the sport of roller derby, a RAG is a team that does not have a formal name but is often referred to as a 'rag team' or just 'RAG.' This term usually applies to a temporary team, a team with an informal name, or a team that is not officially affiliated with the larger roller derby organization.\n\n2. **RAG (Acronym)**: RAG can also be used as an acronym for various purposes. For example, RAG might stand for 'Random Access Graphics,' 'Rapid Application Grid,' or 'Rapid Application Gateway.'", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 40, 'total_tokens': 189, 'completion_time': 0.382776584, 'prompt_time': 0.00328308, 'queue_time': 0.053014599, 'total_time': 0.386059664}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': No